In [1]:
# imports
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

In [2]:
# loading training dataset
df = pd.read_csv("/Users/admin/Desktop/reliable_rejection_under_degradation/verification_mlp/verification_mlp_multilayer/mlp_multilayer_fullExtendedVersion/train_mlp_extended_final_plus_tinyface.csv")

print(df.shape)

df.head()

(5117, 6)


,quality_score,best_similarity,margin,label,true_identity,gallery_identity
0,14.075935,0.248183,0.000138,0,NaN,NaN
1,15.297855,0.220784,0.031821,1,Jason_Momoa,Jason_Momoa
2,16.786978,-0.033517,-0.218560,1,Emma_Watson,Emma_Watson
3,13.985061,0.568350,0.314188,1,NaN,NaN
4,45.461479,0.201896,0.008333,0,NaN,NaN


In [3]:
# features and labels
X = df[
    [
        "quality_score",
        "best_similarity",
        "margin",
    ]
]

y = df["label"]
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True))
print(X.shape)
print(y.shape)

label
1    2582
0    2535
Name: count, dtype: int64
label
1    0.504593
0    0.495407
Name: proportion, dtype: float64
(5117, 3)
(5117,)


In [4]:
# train validation split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(X_train.shape)
print(X_val.shape)

(4093, 3)
(1024, 3)


In [5]:
# scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

joblib.dump(
    scaler,
    "mlp_scaler_finalExp.pkl",
)

print("Scaler Saved")

Scaler Saved


In [6]:
# tensor conversion
X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32,
)

X_val_tensor = torch.tensor(
    X_val_scaled,
    dtype=torch.float32,
)

y_train_tensor = torch.tensor(
    y_train.values,
    dtype=torch.long,
)

y_val_tensor = torch.tensor(
    y_val.values,
    dtype=torch.long,
)

In [7]:
class MLPExperiment(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(3,32),
            nn.ReLU(),

            nn.Linear(32,16),
            nn.ReLU(),

            nn.Linear(16,8),
            nn.ReLU(),

            nn.Linear(8,2)
        )

    def forward(self,x):

        return self.network(x)

In [8]:
# model setup
model = MLPExperiment()

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0005,
    weight_decay=1e-4,
)

print(model)

MLPExperiment(
  (network): Sequential(
    (0): Linear(in_features=3, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=8, bias=True)
    (5): ReLU()
    (6): Linear(in_features=8, out_features=2, bias=True)
  )
)


In [9]:
# training the model
epochs = 300
best_epoch=0
best_val_accuracy = 0

for epoch in range(epochs):

    model.train()

    outputs = model(X_train_tensor)

    loss = criterion(
        outputs,
        y_train_tensor,
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    model.eval()

    with torch.no_grad():

        val_outputs = model(X_val_tensor)

        predictions = torch.argmax(
            val_outputs,
            dim=1,
        )

        val_accuracy = accuracy_score(
            y_val_tensor.numpy(),
            predictions.numpy(),
        )

    if val_accuracy >= best_val_accuracy:

        best_val_accuracy = val_accuracy
        best_epoch=epoch

        torch.save(
            model.state_dict(),
            "mlp_finalExp_model.pth",
        )

    if epoch % 10 == 0:

        print(
            f"Epoch {epoch:3d} | "
            f"Loss {loss.item():.4f} | "
            f"Val Acc {val_accuracy:.4f}"
        )

print()

print(
    "Best Validation Accuracy:",
    best_val_accuracy,
)

Epoch   0 | Loss 0.7046 | Val Acc 0.5049
Epoch  10 | Loss 0.6964 | Val Acc 0.5049
Epoch  20 | Loss 0.6888 | Val Acc 0.5049
Epoch  30 | Loss 0.6808 | Val Acc 0.5049
Epoch  40 | Loss 0.6717 | Val Acc 0.5049
Epoch  50 | Loss 0.6606 | Val Acc 0.5049
Epoch  60 | Loss 0.6465 | Val Acc 0.5049
Epoch  70 | Loss 0.6295 | Val Acc 0.5049
Epoch  80 | Loss 0.6117 | Val Acc 0.5049
Epoch  90 | Loss 0.5915 | Val Acc 0.5049
Epoch 100 | Loss 0.5689 | Val Acc 0.5049
Epoch 110 | Loss 0.5442 | Val Acc 0.5049
Epoch 120 | Loss 0.5184 | Val Acc 0.6436
Epoch 130 | Loss 0.4923 | Val Acc 0.8857
Epoch 140 | Loss 0.4672 | Val Acc 0.8965
Epoch 150 | Loss 0.4435 | Val Acc 0.9023
Epoch 160 | Loss 0.4211 | Val Acc 0.9072
Epoch 170 | Loss 0.4001 | Val Acc 0.9150
Epoch 180 | Loss 0.3802 | Val Acc 0.9160
Epoch 190 | Loss 0.3613 | Val Acc 0.9199
Epoch 200 | Loss 0.3435 | Val Acc 0.9199
Epoch 210 | Loss 0.3274 | Val Acc 0.9209
Epoch 220 | Loss 0.3135 | Val Acc 0.9219
Epoch 230 | Loss 0.3022 | Val Acc 0.9219
Epoch 240 | Loss

In [10]:
# loading the best model
best_model = MLPExperiment()

best_model.load_state_dict(
    torch.load(
        "mlp_finalExp_model.pth",
        map_location=torch.device("cpu"),
    )
)

best_model.eval()

print("Best Model Loaded")

Best Model Loaded


In [11]:
# validation metrics
with torch.no_grad():

    outputs = best_model(X_val_tensor)

    predictions = torch.argmax(
        outputs,
        dim=1,
    )

y_pred = predictions.numpy()

y_true = y_val_tensor.numpy()

In [12]:
# final metrics (acfuracy, precision, F1, Recall)
accuracy = accuracy_score(
    y_true,
    y_pred,
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0,
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0,
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0,
)

print("Accuracy :", accuracy)
print("Best validation accuracy:", best_val_accuracy)
print("Precision:", precision)
print("Best Epoch:", best_epoch)
print("Recall   :", recall)

print("F1 Score :", f1)

Accuracy : 0.92578125
Best validation accuracy: 0.92578125
Precision: 0.9846153846153847
Best Epoch: 244
Recall   : 0.8665377176015474
F1 Score : 0.9218106995884774


In [13]:
# confusion matrix
cm = confusion_matrix(
    y_true,
    y_pred,
)

print(cm)

print()

print(
    classification_report(
        y_true,
        y_pred,
    )
)

[[500   7]
 [ 69 448]]

              precision    recall  f1-score   support

           0       0.88      0.99      0.93       507
           1       0.98      0.87      0.92       517

    accuracy                           0.93      1024
   macro avg       0.93      0.93      0.93      1024
weighted avg       0.93      0.93      0.93      1024



In [14]:
# TRR TAR FRR FAR
TN, FP, FN, TP = cm.ravel()

TAR = TP / (TP + FN)

FRR = FN / (TP + FN)

TRR = TN / (TN + FP)

FAR = FP / (TN + FP)

print()

print("TAR:", TAR)

print("FRR:", FRR)

print("TRR:", TRR)

print("FAR:", FAR)


TAR: 0.8665377176015474
FRR: 0.13346228239845262
TRR: 0.9861932938856016
FAR: 0.013806706114398421


In [16]:
train_df = pd.read_csv("/Users/admin/Desktop/reliable_rejection_under_degradation/verification_mlp/verification_mlp_multilayer/mlp_multilayer_fullExtendedVersion/train_mlp_extended_final_plus_tinyface.csv")

print(train_df[["quality_score","best_similarity","margin"]].describe())

       quality_score  best_similarity       margin
count    5117.000000      5117.000000  5117.000000
mean       20.402349         0.427164     0.178972
std         5.671775         0.213087     0.207421
min        12.476462        -0.063376    -0.664410
25%        16.081200         0.246570     0.016324
50%        19.092539         0.338243     0.074121
75%        23.314095         0.638665     0.375586
max        47.028496         0.892248     0.697794
